In [2]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [4]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

c:\Users\HP\Desktop\LangGraph\myenv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [5]:
from typing import TypedDict

In [6]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)

In [7]:
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000012D04A880D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000012D05BABD50>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [8]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int
    
    sr: float
    bpb: float
    boundary_percent: float
    summary: str

In [9]:
def calculate_sr(state: BatsmanState):
    
    runs = state['runs']
    balls = state['balls']
    
    sr = (runs/balls)*100
    
    
    return {'sr': sr}

In [10]:
def calculate_bpb(state: BatsmanState):
    bpb = state['balls'] / (state['fours'] + state['sixes'])
    
    
    return {'bpb' : bpb}

In [11]:
def calculate_boundary_percent(state: BatsmanState):
    boundary_percent = (((state['fours']*4) + state['sixes']*6) / state['runs']) * 100
    
    
    return {'boundary_percent': boundary_percent} 

In [12]:
def summary(state: BatsmanState) -> BatsmanState:
    summary = f"""
    striken Rate - {state['sr']} \n
    Balls Per Boundary - {state['bpb']} \n
    Boundary_percent - {state['boundary_percent']}
    """
  
    
    return {'summary': summary}

In [13]:
#creat graph
graph = StateGraph(BatsmanState)

#create node
graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('summary', summary)

#create edges
graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_boundary_percent')

graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_bpb', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')

graph.add_edge('summary', END)

#compile graph

workflow = graph.compile()

In [14]:
intial_state = {'runs': 200, 'balls' : 100, 'fours': 12, 'sixes': 9}
final_state = workflow.invoke(intial_state)

In [15]:
print(final_state)

{'runs': 200, 'balls': 100, 'fours': 12, 'sixes': 9, 'sr': 200.0, 'bpb': 4.761904761904762, 'boundary_percent': 51.0, 'summary': '\n    striken Rate - 200.0 \n\n    Balls Per Boundary - 4.761904761904762 \n\n    Boundary_percent - 51.0\n    '}


In [2]:
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())

NameError: name 'workflow' is not defined